# Multi-Basin Rainfall–Runoff LSTM — Revised and Documented

This notebook trains one LSTM model across the ten catchments listed below and evaluates each catchment on its **last five years of daily observations**.

## What this version does differently

1. Fits meteorological and discharge normalization statistics using **training data only**.
2. Computes basin-specific discharge variability from the **training period only**, and uses it on the correct normalized scale in the loss function.
3. Uses a separate temporal validation period and early stopping instead of always taking the final epoch.
4. Balances training samples across catchments so longer records do not dominate merely because they have more rows.
5. Does not backfill missing discharge with a future observation.
6. Checks dates, missing values, incomplete forcing days, and target/forcing alignment before training.
7. Reports NSE, R², RMSE, MAE, percent bias, and KGE by basin; saves predictions and plots for review.

> **Forecast vs. simulation:** `USE_TARGET_DAY_METEOROLOGY = True` (default) means the model estimates discharge for day *t* using meteorological forcing through day *t*. This is a same-day rainfall–runoff simulation/reconstruction, not a strict one-day-ahead forecast. Set it to `False` if you need to predict discharge on day *t* using meteorology only through day *t−1*.

The CSVs must contain the feature columns listed in the configuration cell. Put the ten CSV files in the Colab working directory, or run the upload cell when prompted. The final test years are kept out of training, validation, normalization fitting, and early stopping.

## 1. Configuration
Change these settings in one place. In particular, check the basin filenames and choose the intended forcing/target-day alignment before training.

In [1]:
# -----------------------------
# Basin files and model settings
# -----------------------------
CSV_FILES = [
    "CAN_01AD003.csv",
    "CAN_01BJ003.csv",
    "CAN_02CF007.csv",
    "CAN_05BB001.csv",
    "USA_01013500.csv",
    "USA_01073000.csv",
    "USA_02015700.csv",
    "USA_03346000.csv",
    "USA_04216418.csv",
]

MET_COLS = [
    "Precipitation_mm_day",
    "Temperature_C",
    "Air_Pressure_Pa",
    "Longwave_Rad_W_m2",
    "Shortwave_Rad_W_m2",
    "Specific_Humidity_kg_kg",
    "Wind_Speed_m_s",
]

STATIC_COLS = [
    "elevation",
    "HRUarea",
    "tan_slope",
    "slopeTypeIndex",
    "soilTypeIndex",
    "vegTypeIndex",
    "mHeight",
]

TARGET_COL = "Observed_Discharge_cfs"
DATE_COL_CANDIDATES = ["Timestamp", "Date", "datetime", "time"]
FORCING_HOURS_COL = "Forcing_Hour_Count"  # optional; used to flag incomplete daily forcing

LOOKBACK_DAYS = 365
TEST_DAYS = 5 * 365           # final five years, using daily rows
VALIDATION_DAYS = 3 * 365     # three years immediately before the test period
USE_TARGET_DAY_METEOROLOGY = True

HIDDEN_SIZE = 128
DROPOUT = 0.10
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
MAX_EPOCHS = 40
EARLY_STOPPING_PATIENCE = 6
MIN_VALIDATION_IMPROVEMENT = 1e-4
BATCH_SIZE = 256
GRAD_CLIP_NORM = 1.0

MODEL_PATH = "multibasin_lstm_model_v2.pth"
NORMALIZATION_PATH = "multibasin_norm_stats_v2.npz"
METRICS_PATH = "multibasin_test_metrics_v2.csv"
PREDICTIONS_PATH = "multibasin_test_predictions_v2.csv"


In [2]:
#Pip install packages

In [6]:
%pip install numpy pandas matplotlib scikit-learn ipywidgets ipykernel torch

Looking in links: /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/x86-64-v4, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/x86-64-v3, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/generic, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/generic
Processing /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/generic/scikit_learn-1.8.0+computecanada-cp311-cp311-linux_x86_64.whl
Processing /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/x86-64-v3/torch-2.14.1+computecanada-cp311-cp311-linux_x86_64.whl
Processing /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/generic/scipy-1.17.1+computecanada-cp311-cp311-linux_x86_64.whl (from scikit-learn)
Processing /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/generic/joblib-1.6.0+computecanada-py3-none-any.whl (from scikit-learn)
Processing /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/generic/threadpoolctl-3.7.0+computecanada-py

In [8]:
%pip install --ignore-installed setuptools

Looking in links: /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/x86-64-v4, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/x86-64-v3, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/generic, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/generic
Processing /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/generic/setuptools-84.2.1+computecanada-py3-none-any.whl
Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install torch scikit-learn

Looking in links: /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/x86-64-v4, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/x86-64-v3, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/generic, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/generic
Processing /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/x86-64-v3/torch-2.14.1+computecanada-cp311-cp311-linux_x86_64.whl
Processing /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/generic/scikit_learn-1.8.0+computecanada-cp311-cp311-linux_x86_64.whl
Processing /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/generic/networkx-3.7+computecanada-py3-none-any.whl (from torch)
Processing /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/generic/jinja2-3.1.6+computecanada-py3-none-any.whl (from torch)
Processing /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/generic/fsspec-2026.9.0+computecanada-py3-none-any.whl (from torch)
Processing /c

In [19]:
import numpy
import pandas
import torch
import matplotlib
import sklearn
import ipywidgets

print("All required packages imported successfully!")
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

All required packages imported successfully!
PyTorch version: 2.14.1
GPU available: True


## 2. Imports and reproducibility
Set random seeds and choose the available device. Mixed precision is enabled only when CUDA is available.

In [20]:
import os
import copy
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
# Prefer reproducibility over the fastest possible cuDNN algorithm selection.
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print(f"Device: {device} | Mixed precision: {USE_AMP}")


Device: cuda | Mixed precision: True


## 3. Find or upload the basin CSV files
All ten configured basins are required for the intended multi-basin experiment. If files are missing in Colab, this cell opens an upload dialog for the missing files.

In [21]:
# Look for the expected CSV files in the current notebook directory.
missing_files = [name for name in CSV_FILES if not Path(name).exists()]
if missing_files:
    print("CSV files not yet found:")
    for name in missing_files:
        print(f"  - {name}")
    try:
        from google.colab import files
        print("\nSelect the missing basin CSV files in the upload dialog.")
        files.upload()
    except ImportError:
        print(
            "\nThis notebook is not running in Google Colab. Copy the missing CSV files "
            "into the notebook's working directory and run this cell again."
        )

missing_files = [name for name in CSV_FILES if not Path(name).exists()]
if missing_files:
    raise FileNotFoundError(
        "The following required basin files are missing: " + ", ".join(missing_files)
        + ". Upload all configured CSVs or update CSV_FILES in Section 1."
    )

print(f"Found all {len(CSV_FILES)} basin CSV files.")


Found all 9 basin CSV files.


## 4. Load and audit the data
This step checks required columns, timestamp uniqueness, daily continuity, missing values, static-attribute consistency, and incomplete forcing days. It deliberately does **not** fill missing discharge with a later value. Windows that contain missing meteorological inputs, incomplete daily forcing, or a date gap are excluded from training/evaluation.

In [22]:
def find_date_column(columns):
    """Find a likely date/time field without assuming one exact column spelling."""
    lower_to_original = {str(col).lower(): col for col in columns}
    for candidate in DATE_COL_CANDIDATES:
        if candidate.lower() in lower_to_original:
            return lower_to_original[candidate.lower()]
    for col in columns:
        low = str(col).lower()
        if "time" in low or "date" in low:
            return col
    raise ValueError("Could not find a timestamp/date column.")


def load_and_audit_csv(path):
    """Load one basin file, validate its schema, and retain raw values for later splits."""
    df = pd.read_csv(path)
    date_col = find_date_column(df.columns)

    required = MET_COLS + STATIC_COLS + [TARGET_COL]
    missing_cols = [col for col in required if col not in df.columns]
    if missing_cols:
        raise ValueError(f"{path} is missing required columns: {missing_cols}")

    df = df.copy()
    df["Timestamp"] = pd.to_datetime(df[date_col], errors="coerce")
    if df["Timestamp"].isna().any():
        bad_rows = int(df["Timestamp"].isna().sum())
        raise ValueError(f"{path} has {bad_rows} timestamp value(s) that could not be parsed.")

    df = df.sort_values("Timestamp").reset_index(drop=True)
    duplicate_count = int(df["Timestamp"].duplicated().sum())
    if duplicate_count:
        raise ValueError(f"{path} has {duplicate_count} duplicate timestamp(s); resolve them before training.")

    # Convert model input/target columns to numeric. Unparseable entries become NaN and are audited below.
    for col in MET_COLS + STATIC_COLS + [TARGET_COL]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df[MET_COLS + STATIC_COLS + [TARGET_COL]] = df[MET_COLS + STATIC_COLS + [TARGET_COL]].replace(
        [np.inf, -np.inf], np.nan
    )

    # Static catchment attributes should normally be constant within a basin file.
    varying_static = [col for col in STATIC_COLS if df[col].nunique(dropna=True) > 1]
    if varying_static:
        warnings.warn(
            f"{path}: these static attributes vary over time; the first valid value will be used: {varying_static}"
        )

    # Read one catchment-level value per static attribute and fail if any are missing.
    static_values = {}
    for col in STATIC_COLS:
        nonmissing = df[col].dropna()
        if nonmissing.empty:
            raise ValueError(f"{path}: static attribute {col!r} is entirely missing.")
        static_values[col] = float(nonmissing.iloc[0])

    # Count non-daily gaps. Windows crossing such a gap will be excluded later.
    day_diffs = df["Timestamp"].diff().dt.total_seconds().div(86400).to_numpy()
    gap_flags = np.zeros(len(df), dtype=bool)
    if len(df) > 1:
        gap_flags[1:] = ~np.isclose(day_diffs[1:], 1.0)
    n_gap_edges = int(gap_flags.sum())

    missing_met_count = int(df[MET_COLS].isna().any(axis=1).sum())
    missing_target_count = int(df[TARGET_COL].isna().sum())
    partial_forcing_count = 0
    if FORCING_HOURS_COL in df.columns:
        forcing_hours = pd.to_numeric(df[FORCING_HOURS_COL], errors="coerce")
        partial_forcing_count = int((forcing_hours.notna() & (forcing_hours != 24)).sum())

    raw = {
        "name": Path(path).name,
        "df": df,
        "x_raw": df[MET_COLS].to_numpy(dtype=np.float64),
        "y_raw": df[TARGET_COL].to_numpy(dtype=np.float64),
        "static_raw": np.array([static_values[col] for col in STATIC_COLS], dtype=np.float64),
        "timestamps": df["Timestamp"].to_numpy(dtype="datetime64[ns]"),
        "gap_flags": gap_flags,
        "missing_met_rows": missing_met_count,
        "missing_target_rows": missing_target_count,
        "partial_forcing_rows": partial_forcing_count,
        "gap_edges": n_gap_edges,
    }

    # Make the per-day input-validity flag. Where available, require full 24-hour forcing coverage.
    input_day_valid = np.isfinite(raw["x_raw"]).all(axis=1)
    if FORCING_HOURS_COL in df.columns:
        hours = pd.to_numeric(df[FORCING_HOURS_COL], errors="coerce").to_numpy(dtype=float)
        # A missing or partial Forcing_Hour_Count means the daily met aggregate may be incomplete.
        input_day_valid &= np.isfinite(hours) & np.isclose(hours, 24.0)
    raw["input_day_valid"] = input_day_valid
    return raw


basins = [load_and_audit_csv(path) for path in CSV_FILES]

# A compact audit table helps spot differences before looking at model scores.
audit_rows = []
for basin in basins:
    df = basin["df"]
    valid_q = df[TARGET_COL].dropna()
    annualized_precip = df["Precipitation_mm_day"].mean(skipna=True) * 365.25
    audit_rows.append({
        "Basin": basin["name"],
        "Rows": len(df),
        "Start date": df["Timestamp"].min().date(),
        "End date": df["Timestamp"].max().date(),
        "Missing met rows": basin["missing_met_rows"],
        "Missing discharge rows": basin["missing_target_rows"],
        "Partial forcing rows": basin["partial_forcing_rows"],
        "Non-daily gap edges": basin["gap_edges"],
        "Annualized mean precipitation (mm/year)": annualized_precip,
        "Mean discharge (cfs)": valid_q.mean(),
        "Median discharge (cfs)": valid_q.median(),
        "Max discharge (cfs)": valid_q.max(),
    })

audit_df = pd.DataFrame(audit_rows)
display(audit_df.round(3))

# Show constant catchment attributes; a constant feature is not an error, but provides no
# between-basin discrimination in this particular set of files.
static_matrix = np.vstack([b["static_raw"] for b in basins])
constant_static_cols = [STATIC_COLS[i] for i in range(len(STATIC_COLS)) if np.nanstd(static_matrix[:, i]) < 1e-12]
if constant_static_cols:
    print("Static attributes with no variation across these basins:", constant_static_cols)


,Basin,Rows,Start date,End date,Missing met rows,Missing discharge rows,Partial forcing rows,Non-daily gap edges,Annualized mean precipitation (mm/year),Mean discharge (cfs),Median discharge (cfs),Max discharge (cfs)
0,CAN_01AD003.csv,24930,1951-09-30,2019-12-31,0,1,0,0,999.133,899.711,448.495,18151.704
1,CAN_01BJ003.csv,20181,1964-09-30,2019-12-31,0,1,0,0,1018.220,380.762,140.199,8298.931
2,CAN_02CF007.csv,21734,1960-06-30,2019-12-31,0,25,0,0,826.787,107.030,53.325,3114.748
3,CAN_05BB001.csv,25568,1949-12-31,2019-12-31,0,1,0,0,570.878,1369.688,603.880,16456.604
4,USA_01013500.csv,25568,1949-12-31,2019-12-31,0,1,0,0,979.468,1521.120,824.998,17899.966
5,USA_01073000.csv,25568,1949-12-31,2019-12-31,0,1,2,0,1116.447,20.746,11.000,941.998
6,USA_02015700.csv,21703,1960-07-31,2019-12-31,0,1,2,0,982.902,154.340,84.000,8699.984
7,USA_03346000.csv,25568,1949-12-31,2019-12-31,0,1,2,0,1005.331,295.095,51.000,22499.958
8,USA_04216418.csv,15433,1977-09-30,2019-12-31,0,1,2,0,1052.945,119.528,66.000,3199.994


Static attributes with no variation across these basins: ['tan_slope', 'slopeTypeIndex', 'mHeight']


## 5. Create training, validation, and test periods
For each basin, the final five years are test data; the three years immediately before them are validation data; all earlier rows are training data. The validation years guide early stopping. The test period is not used to fit any scaler or choose the best epoch. The split follows the original notebook’s row-based approach, assuming daily records.

In [23]:
# Use per-basin row indices so records can start/end on different dates.
for basin in basins:
    n = len(basin["df"])
    test_start = n - TEST_DAYS
    val_start = test_start - VALIDATION_DAYS
    train_end = val_start

    earliest_target_idx = LOOKBACK_DAYS - 1 if USE_TARGET_DAY_METEOROLOGY else LOOKBACK_DAYS
    if train_end <= earliest_target_idx:
        raise ValueError(
            f"{basin['name']} has only {n} rows, which is insufficient for the configured "
            f"{LOOKBACK_DAYS}-day lookback, {VALIDATION_DAYS}-day validation period, "
            f"and {TEST_DAYS}-day test period. Reduce these settings or provide more history."
        )

    basin["train_end"] = train_end
    basin["val_start"] = val_start
    basin["test_start"] = test_start

split_rows = []
for basin in basins:
    df = basin["df"]
    split_rows.append({
        "Basin": basin["name"],
        "Training through": df.loc[basin["train_end"] - 1, "Timestamp"].date(),
        "Validation start": df.loc[basin["val_start"], "Timestamp"].date(),
        "Validation end": df.loc[basin["test_start"] - 1, "Timestamp"].date(),
        "Test start": df.loc[basin["test_start"], "Timestamp"].date(),
        "Test end": df.loc[len(df) - 1, "Timestamp"].date(),
    })

display(pd.DataFrame(split_rows))


,Basin,Training through,Validation start,Validation end,Test start,Test end
0,CAN_01AD003.csv,2012-01-02,2012-01-03,2015-01-01,2015-01-02,2019-12-31
1,CAN_01BJ003.csv,2012-01-02,2012-01-03,2015-01-01,2015-01-02,2019-12-31
2,CAN_02CF007.csv,2012-01-02,2012-01-03,2015-01-01,2015-01-02,2019-12-31
3,CAN_05BB001.csv,2012-01-02,2012-01-03,2015-01-01,2015-01-02,2019-12-31
4,USA_01013500.csv,2012-01-02,2012-01-03,2015-01-01,2015-01-02,2019-12-31
5,USA_01073000.csv,2012-01-02,2012-01-03,2015-01-01,2015-01-02,2019-12-31
6,USA_02015700.csv,2012-01-02,2012-01-03,2015-01-01,2015-01-02,2019-12-31
7,USA_03346000.csv,2012-01-02,2012-01-03,2015-01-01,2015-01-02,2019-12-31
8,USA_04216418.csv,2012-01-02,2012-01-03,2015-01-01,2015-01-02,2019-12-31


## 6. Fit normalization statistics on training data only
All meteorological means/scales, global discharge mean/scale, and per-basin discharge standard deviations are estimated from the training section only. Static attributes are catchment metadata, so one value per basin is used. A constant feature gets scale 1 rather than dividing by a near-zero number.

In [24]:
# Pool only the training rows from every basin when fitting dynamic and target scalers.
train_x_parts = []
train_y_parts = []
train_static_parts = []
basin_train_std_cfs = {}

for basin in basins:
    train_end = basin["train_end"]
    train_x_parts.append(basin["x_raw"][:train_end])
    train_y_parts.append(basin["y_raw"][:train_end])
    train_static_parts.append(basin["static_raw"])

    y_train = basin["y_raw"][:train_end]
    y_train = y_train[np.isfinite(y_train)]
    if len(y_train) < 2:
        raise ValueError(f"{basin['name']} has fewer than two valid training discharge values.")
    # ddof=0 is stable and consistent with the global np.nanstd calculation below.
    basin_train_std_cfs[basin["name"]] = float(np.std(y_train, ddof=0))

train_x_all = np.vstack(train_x_parts)
train_y_all = np.concatenate(train_y_parts)
train_static_all = np.vstack(train_static_parts)

mean_x = np.nanmean(train_x_all, axis=0)
std_x = np.nanstd(train_x_all, axis=0)
std_x = np.where(np.isfinite(std_x) & (std_x > 1e-8), std_x, 1.0)
if not np.isfinite(mean_x).all():
    bad_features = [MET_COLS[i] for i, value in enumerate(mean_x) if not np.isfinite(value)]
    raise ValueError(f"These meteorological features have no valid training data: {bad_features}")

finite_train_y = train_y_all[np.isfinite(train_y_all)]
if len(finite_train_y) < 2:
    raise ValueError("Not enough valid training discharge values across basins.")
mean_y = float(np.mean(finite_train_y))
std_y = float(np.std(finite_train_y, ddof=0))
if not np.isfinite(std_y) or std_y <= 1e-8:
    raise ValueError("Global training discharge standard deviation is zero or invalid.")

mean_c = np.nanmean(train_static_all, axis=0)
std_c = np.nanstd(train_static_all, axis=0)
std_c = np.where(np.isfinite(std_c) & (std_c > 1e-8), std_c, 1.0)
if not np.isfinite(mean_c).all():
    raise ValueError("One or more static attributes are invalid across the catchments.")

# Convert the raw per-basin standard deviations into the same scale as globally normalized y.
basin_std_scaled = {
    name: max(std_cfs / std_y, 1e-3)
    for name, std_cfs in basin_train_std_cfs.items()
}

# Pre-compute normalized arrays, while retaining raw observations for reporting and plots.
for basin in basins:
    basin["x_norm"] = ((basin["x_raw"] - mean_x) / std_x).astype(np.float32)
    basin["y_norm"] = ((basin["y_raw"] - mean_y) / std_y).astype(np.float32)
    basin["c_norm"] = ((basin["static_raw"] - mean_c) / std_c).astype(np.float32)

normalization_summary = pd.DataFrame({
    "Meteorological feature": MET_COLS,
    "Training mean": mean_x,
    "Training scale": std_x,
})
display(normalization_summary.round(6))
print(f"Global discharge training mean: {mean_y:.6g} cfs")
print(f"Global discharge training standard deviation: {std_y:.6g} cfs")
print("Per-basin discharge standard deviations were computed from each basin's training period only.")


,Meteorological feature,Training mean,Training scale
0,Precipitation_mm_day,2.571428,5.443066
1,Temperature_C,5.311869,11.626245
2,Air_Pressure_Pa,95268.073789,6933.206976
3,Longwave_Rad_W_m2,285.500133,60.930152
4,Shortwave_Rad_W_m2,159.985979,92.706249
5,Specific_Humidity_kg_kg,0.005254,0.003715
6,Wind_Speed_m_s,2.288610,0.965021


Global discharge training mean: 584.105 cfs
Global discharge training standard deviation: 1188.51 cfs
Per-basin discharge standard deviations were computed from each basin's training period only.


## 7. Build valid lookback windows
The dataset creates windows on demand rather than copying every 365-day window into RAM. A sample is skipped if its target discharge is missing, any meteorological input in the window is missing/incomplete, or the window crosses a non-daily timestamp gap. The loss compares errors on a basin-standard-deviation scale that is consistent with the globally normalized target.

In [25]:
class MultiBasinSequenceDataset(Dataset):
    """Generate (meteorology window, static attributes, target) samples lazily."""

    def __init__(self, basin_records, split_name):
        if split_name not in {"train", "validation", "test"}:
            raise ValueError("split_name must be 'train', 'validation', or 'test'.")
        self.basin_records = basin_records
        self.split_name = split_name
        self.samples = []
        self.counts = []

        for basin_idx, basin in enumerate(basin_records):
            if split_name == "train":
                target_start, target_end = 0, basin["train_end"]
            elif split_name == "validation":
                target_start, target_end = basin["val_start"], basin["test_start"]
            else:
                target_start, target_end = basin["test_start"], len(basin["df"])

            first_possible_target = LOOKBACK_DAYS - 1 if USE_TARGET_DAY_METEOROLOGY else LOOKBACK_DAYS
            target_start = max(target_start, first_possible_target)

            # Prefix sums make the window-validity checks efficient.
            input_day_valid = basin["input_day_valid"]
            invalid_prefix = np.concatenate([[0], np.cumsum(~input_day_valid)])
            gap_prefix = np.concatenate([[0], np.cumsum(basin["gap_flags"])])
            valid_count = 0
            skipped_missing_target = 0
            skipped_bad_window = 0

            for t in range(target_start, target_end):
                # Same-day simulation includes t; forecast mode uses only rows strictly before t.
                start = t - LOOKBACK_DAYS + 1 if USE_TARGET_DAY_METEOROLOGY else t - LOOKBACK_DAYS
                end = t + 1 if USE_TARGET_DAY_METEOROLOGY else t
                if start < 0 or end - start != LOOKBACK_DAYS:
                    continue
                if not np.isfinite(basin["y_raw"][t]):
                    skipped_missing_target += 1
                    continue
                # Every met day in the input sequence must be complete and finite.
                if invalid_prefix[end] - invalid_prefix[start] > 0:
                    skipped_bad_window += 1
                    continue
                # gap_flags[i] marks a gap between i-1 and i. Ignore a gap immediately before window start.
                if end - start > 1 and gap_prefix[end] - gap_prefix[start + 1] > 0:
                    skipped_bad_window += 1
                    continue

                self.samples.append((basin_idx, t))
                valid_count += 1

            self.counts.append({
                "Basin": basin["name"],
                "Split": split_name,
                "Valid samples": valid_count,
                "Skipped missing targets": skipped_missing_target,
                "Skipped incomplete/gapped windows": skipped_bad_window,
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        basin_idx, t = self.samples[index]
        basin = self.basin_records[basin_idx]
        start = t - LOOKBACK_DAYS + 1 if USE_TARGET_DAY_METEOROLOGY else t - LOOKBACK_DAYS
        end = t + 1 if USE_TARGET_DAY_METEOROLOGY else t
        x_window = basin["x_norm"][start:end]
        y_value = basin["y_norm"][t]
        return {
            "x": torch.from_numpy(x_window),
            "c": torch.from_numpy(basin["c_norm"]),
            "y": torch.tensor([y_value], dtype=torch.float32),
            "basin_std_scaled": torch.tensor([basin_std_scaled[basin["name"]]], dtype=torch.float32),
            "basin_idx": torch.tensor(basin_idx, dtype=torch.long),
        }


train_dataset = MultiBasinSequenceDataset(basins, "train")
val_dataset = MultiBasinSequenceDataset(basins, "validation")
test_dataset = MultiBasinSequenceDataset(basins, "test")

sample_audit = pd.DataFrame(train_dataset.counts + val_dataset.counts + test_dataset.counts)
display(sample_audit)

if len(train_dataset) == 0 or len(val_dataset) == 0 or len(test_dataset) == 0:
    raise ValueError(
        "At least one split has no valid samples. Check the file date ranges, missing values, "
        "incomplete forcing days, lookback, and split lengths."
    )


,Basin,Split,Valid samples,Skipped missing targets,Skipped incomplete/gapped windows
0,CAN_01AD003.csv,train,21646,0,0
1,CAN_01BJ003.csv,train,16897,0,0
2,CAN_02CF007.csv,train,18450,0,0
3,CAN_05BB001.csv,train,22284,0,0
4,USA_01013500.csv,train,22284,0,0
5,USA_01073000.csv,train,22283,0,1
6,USA_02015700.csv,train,18418,0,1
7,USA_03346000.csv,train,22283,0,1
8,USA_04216418.csv,train,12148,0,1
9,CAN_01AD003.csv,validation,1095,0,0


## 8. Create basin-balanced data loaders
The weighted sampler draws training examples so each basin contributes approximately equally per epoch. This avoids a longer basin dominating training solely because it has more valid windows. Validation and test loaders are deterministic and unshuffled.

In [26]:
# Each training sample gets weight 1 / (valid samples in its basin).
# Therefore, the total sampling weight is approximately equal for every basin.
train_counts = {}
for basin_idx, _ in train_dataset.samples:
    train_counts[basin_idx] = train_counts.get(basin_idx, 0) + 1
sample_weights = [1.0 / train_counts[basin_idx] for basin_idx, _ in train_dataset.samples]
train_sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_dataset),
    replacement=True,
)

loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": 0,  # robust in Colab; windows are generated lazily
    "pin_memory": device.type == "cuda",
}
train_loader = DataLoader(train_dataset, sampler=train_sampler, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

print(f"Training windows: {len(train_dataset):,}")
print(f"Validation windows: {len(val_dataset):,}")
print(f"Test windows: {len(test_dataset):,}")


Training windows: 176,693
Validation windows: 9,855
Test windows: 16,421


In [28]:
# step 9 problem fix

import torch

# Work around a possible incompatibility in the cuDNN LSTM implementation.
# This disables cuDNN, but does not disable CUDA itself.
torch.backends.cudnn.enabled = False

# Continue using the GPU if it is available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
PyTorch: 2.14.1
CUDA build: 13.2
GPU: Tesla T4


In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU capability:", torch.cuda.get_device_capability(0))
    print("Architectures in PyTorch:", torch.cuda.get_arch_list())

    # Test basic GPU operations without the LSTM
    try:
        x = torch.randn(2, 3, device="cuda")
        y = torch.cat([x, x], dim=0)
        torch.cuda.synchronize()
        print("GPU tensor test passed:", y.shape)
    except Exception as e:
        print("GPU tensor test failed:", repr(e))

PyTorch version: 2.14.1
CUDA build: 13.2
CUDA available: True
GPU: Tesla T4
GPU capability: (7, 5)
Architectures in PyTorch: ['sm_80', 'sm_90', 'sm_100', 'compute_100']
GPU tensor test failed: AcceleratorError("CUDA error: no kernel image is available for execution on the device\nSearch for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.\n")


/localscratch/pollen26.25118650.0/lib/python3.11/site-packages/torch/cuda/__init__.py:484: UserWarning: Found GPU0 Tesla T4 which is of compute capability (CC) 7.5.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 8.0 which supports hardware CC >=8.0,<9.0
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
Your installed torch==2.14.1 does not include kernels for this GPU. Reinstall the same version against a CUDA build that does, e.g.:
  For CUDA 12.6 use pip install torch==2.14.1 --index-url https://download.pytorch.org/whl/cu126
  For CUDA 13.0 use pip install torch==2.14.1 --index-url https://download.pytorch.org/whl/cu130
  For CUDA 13.2 use pip install torch==2.14.1 --index-url https://download.pytorch.org/whl/cu132
  For CUDA 13.4 use pip install torch==2.14.1 --index-url https://download.pytorch.org/whl/cu13

## 9. Define the LSTM and corrected loss
The network receives the dynamic meteorological sequence and the catchment attributes repeated across the sequence. The loss is a **basin-standardized MSE**: target and prediction are both on the global normalized scale, so the basin standard deviation is converted to that same scale before division. The basin standard deviations were computed from the training years only.

In [29]:
class MultiBasinLSTM(nn.Module):
    """LSTM rainfall-runoff model conditioned on static catchment attributes."""

    def __init__(self, dynamic_dim, static_dim, hidden_size=128, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=dynamic_dim + static_dim,
            hidden_size=hidden_size,
            batch_first=True,
        )
        self.dropout = nn.Dropout(p=dropout)
        self.output_layer = nn.Linear(hidden_size, 1)

    def forward(self, x_dynamic, c_static):
        # Repeat catchment attributes along the lookback dimension, then concatenate with met inputs.
        c_repeated = c_static.unsqueeze(1).expand(-1, x_dynamic.shape[1], -1)
        x_combined = torch.cat([x_dynamic, c_repeated], dim=-1)
        sequence_output, _ = self.lstm(x_combined)
        final_state = sequence_output[:, -1, :]
        return self.output_layer(self.dropout(final_state))


def basin_standardized_mse(prediction, target, basin_std_scaled):
    """Compare errors after scaling each basin's error by its training-period variability."""
    return torch.mean(((prediction - target) / basin_std_scaled.clamp_min(1e-3)) ** 2)


model = MultiBasinLSTM(
    dynamic_dim=len(MET_COLS),
    static_dim=len(STATIC_COLS),
    hidden_size=HIDDEN_SIZE,
    dropout=DROPOUT,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
try:
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
except (AttributeError, TypeError):
    # Fallback for older Colab/PyTorch versions.
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print(model)


MultiBasinLSTM(
  (lstm): LSTM(14, 128, batch_first=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (output_layer): Linear(in_features=128, out_features=1, bias=True)
)


## 10. Train with validation-based early stopping
Each epoch records a training loss and a basin-balanced validation loss. The best validation model is retained, and training stops when the validation score fails to improve for the configured patience.

In [30]:
def run_validation_loss(model, loader):
    """Return average per-basin standardized MSE, giving each basin equal validation weight."""
    model.eval()
    errors_by_basin = {i: [] for i in range(len(basins))}
    with torch.no_grad():
        for batch in loader:
            x = batch["x"].to(device, non_blocking=True)
            c = batch["c"].to(device, non_blocking=True)
            y = batch["y"].to(device, non_blocking=True)
            basin_std = batch["basin_std_scaled"].to(device, non_blocking=True)
            basin_idx = batch["basin_idx"].cpu().numpy()
            with torch.autocast(device_type=device.type, enabled=USE_AMP):
                pred = model(x, c)
            values = (((pred.float() - y.float()) / basin_std.float().clamp_min(1e-3)) ** 2).squeeze(-1)
            values = values.cpu().numpy()
            for b_idx, value in zip(basin_idx, values):
                errors_by_basin[int(b_idx)].append(float(value))

    per_basin = [np.mean(v) for v in errors_by_basin.values() if len(v) > 0]
    return float(np.mean(per_basin)) if per_basin else np.inf


history = {"epoch": [], "train_loss": [], "validation_loss": []}
best_val_loss = np.inf
best_state = None
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    train_loss_sum = 0.0
    n_seen = 0

    for batch in train_loader:
        x = batch["x"].to(device, non_blocking=True)
        c = batch["c"].to(device, non_blocking=True)
        y = batch["y"].to(device, non_blocking=True)
        basin_std = batch["basin_std_scaled"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=USE_AMP):
            pred = model(x, c)
            loss = basin_standardized_mse(pred, y, basin_std)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()

        batch_n = x.shape[0]
        train_loss_sum += float(loss.detach().cpu()) * batch_n
        n_seen += batch_n

    train_loss = train_loss_sum / max(n_seen, 1)
    val_loss = run_validation_loss(model, val_loader)
    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["validation_loss"].append(val_loss)

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Training standardized MSE: {train_loss:.6f} | "
        f"Basin-balanced validation MSE: {val_loss:.6f}"
    )

    if val_loss < best_val_loss - MIN_VALIDATION_IMPROVEMENT:
        best_val_loss = val_loss
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping: validation did not improve for {EARLY_STOPPING_PATIENCE} epochs.")
        break

if best_state is None:
    raise RuntimeError("Training did not produce a valid checkpoint.")
model.load_state_dict(best_state)
model.eval()
print(f"\nRestored best model from epoch {best_epoch} (validation loss={best_val_loss:.6f}).")


AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.


## 11. Review the training curve
The best epoch is chosen by validation loss, not by the final test scores. A falling training loss with rising validation loss can indicate overfitting; both remaining high can indicate underfitting or insufficient input information.

In [ ]:
history_df = pd.DataFrame(history)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Training loss")
ax.plot(history_df["epoch"], history_df["validation_loss"], marker="o", label="Validation loss (basin-balanced)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Basin-standardized MSE")
ax.set_title("Training and validation curves")
ax.grid(True, linestyle=":", alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()


## 12. Evaluate the untouched final five years
Predictions are converted back to cfs using the global discharge scale fitted on the training years. Metrics are calculated only for valid target days. Positive percent bias means the model overpredicts total discharge; negative means underprediction.

In [ ]:
def calculate_nse(obs, pred):
    denominator = np.sum((obs - np.mean(obs)) ** 2)
    return float(1.0 - np.sum((obs - pred) ** 2) / denominator) if denominator > 0 else np.nan


def calculate_kge(obs, pred):
    """Kling-Gupta efficiency (2009 form); NaN if its components are undefined."""
    if len(obs) < 2 or np.std(obs) == 0 or np.std(pred) == 0 or np.mean(obs) == 0:
        return np.nan
    corr = np.corrcoef(obs, pred)[0, 1]
    alpha = np.std(pred, ddof=0) / np.std(obs, ddof=0)
    beta = np.mean(pred) / np.mean(obs)
    if not np.isfinite([corr, alpha, beta]).all():
        return np.nan
    return float(1.0 - np.sqrt((corr - 1.0) ** 2 + (alpha - 1.0) ** 2 + (beta - 1.0) ** 2))


# Generate predictions for valid test samples. DataLoader order is deterministic (no shuffle).
model.eval()
pred_scaled_parts = []
target_scaled_parts = []
basin_idx_parts = []
target_idx_parts = []
with torch.no_grad():
    sample_offset = 0
    for batch in test_loader:
        x = batch["x"].to(device, non_blocking=True)
        c = batch["c"].to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=USE_AMP):
            pred_scaled = model(x, c)
        pred_scaled_parts.append(pred_scaled.float().cpu().numpy().reshape(-1))
        target_scaled_parts.append(batch["y"].numpy().reshape(-1))
        basin_idx_parts.extend(batch["basin_idx"].numpy().astype(int).tolist())
        # Recover (basin index, target row index) from the dataset's deterministic sample list.
        batch_n = len(batch["basin_idx"])
        batch_samples = test_dataset.samples[sample_offset:sample_offset + batch_n]
        target_idx_parts.extend([t for _, t in batch_samples])
        sample_offset += batch_n

pred_scaled_all = np.concatenate(pred_scaled_parts)
target_scaled_all = np.concatenate(target_scaled_parts)
pred_cfs_all = pred_scaled_all * std_y + mean_y
obs_cfs_all = target_scaled_all * std_y + mean_y

# Assemble per-basin prediction frames and metrics.
all_results = {}
metric_rows = []
for basin_idx, basin in enumerate(basins):
    local_positions = [i for i, b_idx in enumerate(basin_idx_parts) if b_idx == basin_idx]
    local_target_indices = [target_idx_parts[i] for i in local_positions]
    local_pred = pred_cfs_all[local_positions]
    local_obs = obs_cfs_all[local_positions]

    result_df = basin["df"].iloc[local_target_indices][["Timestamp", TARGET_COL, "Precipitation_mm_day"]].copy()
    result_df = result_df.rename(columns={TARGET_COL: "Observed_Discharge_cfs"})
    result_df["Predicted_Discharge_cfs"] = local_pred
    result_df["Basin"] = basin["name"]
    all_results[basin["name"]] = result_df.reset_index(drop=True)

    obs = result_df["Observed_Discharge_cfs"].to_numpy(dtype=float)
    pred = result_df["Predicted_Discharge_cfs"].to_numpy(dtype=float)
    residual = pred - obs
    denominator = np.sum(obs)
    pbias = float(100.0 * np.sum(residual) / denominator) if abs(denominator) > 1e-12 else np.nan

    metric_rows.append({
        "Catchment": basin["name"],
        "Test Start Date": result_df["Timestamp"].min().date() if len(result_df) else None,
        "Test End Date": result_df["Timestamp"].max().date() if len(result_df) else None,
        "Valid Test Days": len(result_df),
        "NSE": calculate_nse(obs, pred) if len(obs) else np.nan,
        # R² uses the coefficient-of-determination definition (same formula as NSE here).
        "R²": calculate_nse(obs, pred) if len(obs) else np.nan,
        "KGE": calculate_kge(obs, pred) if len(obs) else np.nan,
        "RMSE (cfs)": float(np.sqrt(np.mean(residual ** 2))) if len(obs) else np.nan,
        "MAE (cfs)": float(np.mean(np.abs(residual))) if len(obs) else np.nan,
        "PBIAS (%)": pbias,
        "Mean observed (cfs)": float(np.mean(obs)) if len(obs) else np.nan,
        "Mean predicted (cfs)": float(np.mean(pred)) if len(pred) else np.nan,
    })

summary_df = pd.DataFrame(metric_rows).sort_values("NSE", ascending=True, na_position="last").reset_index(drop=True)
print("Test performance (lowest NSE first):")
display(summary_df.round(4))


## 13. Save predictions, metrics, normalization statistics, and model
The exported metrics and predictions are intended for downstream analysis. The checkpoint records the feature names and key settings needed to interpret it. Normalization statistics contain training-only values.

In [ ]:
# Combine the individual test predictions into one file.
if all_results:
    combined_predictions = pd.concat(all_results.values(), ignore_index=True)
else:
    combined_predictions = pd.DataFrame()

summary_df.to_csv(METRICS_PATH, index=False)
combined_predictions.to_csv(PREDICTIONS_PATH, index=False)

np.savez(
    NORMALIZATION_PATH,
    mean_x=mean_x,
    std_x=std_x,
    mean_c=mean_c,
    std_c=std_c,
    mean_y=np.array(mean_y),
    std_y=np.array(std_y),
    met_cols=np.array(MET_COLS),
    static_cols=np.array(STATIC_COLS),
    basin_names=np.array([b["name"] for b in basins]),
    basin_train_std_cfs=np.array([basin_train_std_cfs[b["name"]] for b in basins]),
)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "dynamic_dim": len(MET_COLS),
    "static_dim": len(STATIC_COLS),
    "hidden_size": HIDDEN_SIZE,
    "dropout": DROPOUT,
    "met_cols": MET_COLS,
    "static_cols": STATIC_COLS,
    "target_col": TARGET_COL,
    "lookback_days": LOOKBACK_DAYS,
    "use_target_day_meteorology": USE_TARGET_DAY_METEOROLOGY,
    "best_epoch": best_epoch,
    "best_validation_loss": best_val_loss,
    "basin_train_std_cfs": basin_train_std_cfs,
}
torch.save(checkpoint, MODEL_PATH)

print("Saved files:")
for path in [MODEL_PATH, NORMALIZATION_PATH, METRICS_PATH, PREDICTIONS_PATH]:
    print(f"  {path} ({Path(path).stat().st_size:,} bytes)")


## 14. Plot observed vs. predicted hydrographs
Use this helper to visually inspect any basin or date range. Hydrographs are especially useful for identifying peak underestimation, timing errors, seasonal bias, and low-flow mismatch that a single NSE value cannot explain.

In [ ]:
def plot_basin_hydrograph(basin_name, start_date=None, end_date=None, show_precipitation=True):
    """Plot observations and model predictions for one basin within an optional date range."""
    if basin_name not in all_results:
        raise KeyError(f"Unknown basin {basin_name!r}. Choose one of: {list(all_results)}")
    frame = all_results[basin_name].copy()
    if start_date is not None:
        frame = frame[frame["Timestamp"] >= pd.Timestamp(start_date)]
    if end_date is not None:
        frame = frame[frame["Timestamp"] <= pd.Timestamp(end_date)]
    if frame.empty:
        print("No valid test predictions in the requested date range.")
        return

    fig, ax1 = plt.subplots(figsize=(15, 5.5))
    ax1.plot(frame["Timestamp"], frame["Observed_Discharge_cfs"], label="Observed discharge", linewidth=1.2)
    ax1.plot(frame["Timestamp"], frame["Predicted_Discharge_cfs"], label="Predicted discharge", linewidth=1.1, linestyle="--")
    ax1.set_xlabel("Date")
    ax1.set_ylabel("Discharge (cfs)")
    ax1.grid(True, linestyle=":", alpha=0.4)
    ax1.legend(loc="upper left")

    if show_precipitation:
        ax2 = ax1.twinx()
        ax2.bar(frame["Timestamp"], frame["Precipitation_mm_day"], width=1.0, alpha=0.22, label="Precipitation")
        ax2.set_ylabel("Precipitation (mm/day)")
        ax2.set_ylim(max(float(frame["Precipitation_mm_day"].max()), 1.0) * 3.0, 0)

    ax1.set_title(f"{basin_name}: observed vs predicted discharge")
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


# Plot the weakest NSE basin by default. You can call the function again with another basin name.
if len(summary_df) and pd.notna(summary_df.loc[0, "NSE"]):
    weakest_basin = summary_df.loc[0, "Catchment"]
    print(f"Lowest test NSE: {weakest_basin}")
    plot_basin_hydrograph(weakest_basin)


## 15. How to interpret the outputs

- **NSE**: 1 is perfect; 0 means performance comparable to predicting the observed mean; negative values are worse than that baseline.
- **R²**: squared correlation, describing association rather than bias or peak accuracy. Read it together with NSE and PBIAS.
- **KGE**: combines correlation, variability ratio, and mean ratio; values near 1 are better.
- **RMSE and MAE**: absolute errors in cfs; their sizes are not directly comparable between differently sized catchments.
- **PBIAS**: positive means total predicted discharge is too high; negative means too low.

### Important checks before comparing this run with the previous notebook

1. This notebook uses a three-year validation period, early stopping, training-only scalers and basin standard deviations, and basin-balanced sampling. It is not an exact reproduction of the earlier training recipe.
2. By default, target-day meteorology is included, so this is a same-day rainfall–runoff simulation. For a strict one-day-ahead forecast, set `USE_TARGET_DAY_METEOROLOGY = False` and rerun the notebook from the top.
3. Incomplete forcing days are not used in windows when `Forcing_Hour_Count` is present. Missing discharge is not imputed; rows with missing targets are skipped.
4. The final five years remain the held-out test set. Do not use test NSE to tune hyperparameters repeatedly; use validation loss for model selection.